# 02 — Data preparation evidence

Preparation and validation evidence for the canonical geometry, governed CLC derivatives, and national panel. National preparation logic lives in reusable `src/` modules and scripts; this notebook does not modify raw or processed data.

In [1]:
from pathlib import Path
import json
import sys
import pyogrio
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_support import resolve_project_root

PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)

from src.config import CLC, SPATIAL, TEMPORAL
from src.feature_contract import FIELD_CONTRACTS, PREDICTOR_COLUMNS, TARGET_COLUMN
from src.geospatial_utils import GRID_PATH
from src.source_registry import CLC_PREPARED_PORTUGAL_LAYERS
print(SPATIAL)

SpatialConfig(analysis_crs='EPSG:3763', grid_size_metres=1000, context_buffer_metres=2000)


## Final model feature contract

This is the single nine-predictor analytical contract used by the national panel and final saved model. All five climate fields, including the two T-only monthly summaries, are derived from JJAS ERA5-Land values in predictor year T. Geometry remains in the canonical grid GeoPackage rather than being repeated in every cell-year row.

In [2]:
feature_contract = pd.DataFrame([
    {'column': name, 'role': 'predictor', 'unit': FIELD_CONTRACTS[name].unit, 'allowed range': f'{FIELD_CONTRACTS[name].minimum} to {FIELD_CONTRACTS[name].maximum}', 'source-year rule': FIELD_CONTRACTS[name].source_year_rule}
    for name in PREDICTOR_COLUMNS
])
display(feature_contract)
assert len(PREDICTOR_COLUMNS) == 9
print('Final saved model feature count:', len(PREDICTOR_COLUMNS))
print('Unique analytical key: cell_id × observation_year; target:', TARGET_COLUMN)

,column,role,unit,allowed range,source-year rule
0,built_up_share,predictor,share_of_cell_land_area,0.0 to 1.0,governed CLC reference year assigned to T
1,forest_shrub_share_2km,predictor,share_of_mainland_land_in_2km_outward_buffer,0.0 to 1.0,governed CLC reference year assigned to T
2,mean_slope_2km,predictor,degrees,0.0 to 90.0,static Copernicus DEM GLO-30 2021 release
3,fire_years_previous_10y_2km,predictor,count_of_distinct_years,0.0 to 10.0,inclusive T-10 through T-1
4,warm_season_mean_2m_temperature_c,predictor,degrees_Celsius,-20.0 to 60.0,JJAS of T only; ERA5-Land air temperature at 2...
5,warm_season_total_precipitation_mm,predictor,millimetres_JJAS_total,0.0 to 3000.0,day-weighted JJAS of T only
6,warm_season_mean_soil_water_layer1,predictor,m3_per_m3,0.0 to 1.0,JJAS of T only
7,warm_season_max_monthly_2m_temperature_c,predictor,degrees_Celsius,-20.0 to 60.0,maximum of monthly-mean June-September air tem...
8,warm_season_min_monthly_soil_water_layer1,predictor,m3_per_m3,0.0 to 1.0,minimum of monthly-mean June-September layer-1...


Final saved model feature count: 9
Unique analytical key: cell_id × observation_year; target: burned_share_next_year


## Canonical grid geometry

In [3]:
grid_info = pyogrio.read_info(GRID_PATH, layer='canonical_mainland_grid_1km')
assert grid_info['features'] == 89_112
assert grid_info['crs'] == 'EPSG:3763'
print({'path': GRID_PATH.relative_to(PROJECT_ROOT).as_posix(), 'layer': 'canonical_mainland_grid_1km', 'features': grid_info['features'], 'crs': grid_info['crs']})

{'path': 'data/processed/reference/canonical_mainland_grid_1km.gpkg', 'layer': 'canonical_mainland_grid_1km', 'features': 89112, 'crs': 'EPSG:3763'}


## Governed Portugal CLC layers

In [4]:
for year, record in CLC_PREPARED_PORTUGAL_LAYERS.items():
    path = PROJECT_ROOT / record.prepared_path
    facts = record.validation_facts
    info = pyogrio.read_info(path, layer=facts.layer_name)
    assert info['features'] == facts.feature_count
    assert info['crs'] == record.crs
    assert facts.class_code_field in info['fields']
    print(year, record.prepared_path, info['features'], info['crs'], 'registered validation: passed')

2006 data/processed/clc/u2012_clc2006_v2020_20u1_pt.gpkg 51555 EPSG:3035 registered validation: passed
2012 data/processed/clc/u2018_clc2012_v2020_20u1_pt.gpkg 54041 EPSG:3035 registered validation: passed
2018 data/processed/clc/u2018_clc2018_v2020_20u1_pt.gpkg 54191 EPSG:3035 registered validation: passed


## CLC assignment and no-future-information check

CLC is retrospective broad landscape context. The assigned reference year must never be later than predictor year T; this table exposes the governed assignment rather than assuming annual land-cover change.

In [5]:
clc_assignment = pd.DataFrame([
    {'predictor year T': year, 'CLC reference year': CLC.reference_year(year), 'prepared path': CLC.prepared_dataset(year)[0], 'layer': CLC.prepared_dataset(year)[1]}
    for year in range(TEMPORAL.predictor_start_year, TEMPORAL.predictor_end_year + 1)
])
assert (clc_assignment['CLC reference year'] <= clc_assignment['predictor year T']).all()
display(clc_assignment)

,predictor year T,CLC reference year,prepared path,layer
0,2015,2006,data/processed/clc/u2012_clc2006_v2020_20u1_pt...,u2012_clc2006_v2020_20u1_pt
1,2016,2012,data/processed/clc/u2018_clc2012_v2020_20u1_pt...,u2018_clc2012_v2020_20u1_pt
2,2017,2012,data/processed/clc/u2018_clc2012_v2020_20u1_pt...,u2018_clc2012_v2020_20u1_pt
3,2018,2012,data/processed/clc/u2018_clc2012_v2020_20u1_pt...,u2018_clc2012_v2020_20u1_pt
4,2019,2018,data/processed/clc/u2018_clc2018_v2020_20u1_pt...,u2018_clc2018_v2020_20u1_pt
5,2020,2018,data/processed/clc/u2018_clc2018_v2020_20u1_pt...,u2018_clc2018_v2020_20u1_pt
6,2021,2018,data/processed/clc/u2018_clc2018_v2020_20u1_pt...,u2018_clc2018_v2020_20u1_pt
7,2022,2018,data/processed/clc/u2018_clc2018_v2020_20u1_pt...,u2018_clc2018_v2020_20u1_pt
8,2023,2018,data/processed/clc/u2018_clc2018_v2020_20u1_pt...,u2018_clc2018_v2020_20u1_pt
9,2024,2018,data/processed/clc/u2018_clc2018_v2020_20u1_pt...,u2018_clc2018_v2020_20u1_pt


## Optional bounded preparation orchestration

The checks above use the actual canonical grid and panel artifacts. The switches below call the same restartable preparation functions used by the scripts; they are intentionally disabled because they rebuild derived components and can take substantial time and memory.

In [6]:
REBUILD_NATIONAL_PANEL = False
REBUILD_EXTENDED_TRAINING_PANEL = False
if REBUILD_NATIONAL_PANEL:
    from src.national_panel import run_national_build
    national_build = run_national_build()
    print('Canonical national panel rebuild completed:', national_build['panel_readiness_decision'])
if REBUILD_EXTENDED_TRAINING_PANEL:
    from src.extended_training_panel import run_extended_panel_build
    extended_build = run_extended_panel_build()
    print('Extended training-panel rebuild completed:', extended_build['actual_row_count'])
if not (REBUILD_NATIONAL_PANEL or REBUILD_EXTENDED_TRAINING_PANEL):
    print('Rebuild switches are disabled. Preferred full automation: python scripts/run_project.py --mode reproduce --confirm-rebuild')

Rebuild switches are disabled. Preferred full automation: python scripts/run_project.py --mode reproduce --confirm-rebuild


## National-panel validation evidence

These stored metrics verify the assembled analytical table before it is used for EDA or modelling: every canonical mainland cell appears once for each observation year, with stable keys and the documented temporal source rules. The detailed validation report remains the authoritative audit record.

In [7]:
metrics_path = PROJECT_ROOT / 'data/processed/national_panel_2015_2024_validation.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
panel_summary = pd.DataFrame([
    {'check': 'canonical grid cells', 'value': metrics['grid_cell_count']},
    {'check': 'expected cell-year rows', 'value': metrics['expected_row_count']},
    {'check': 'actual cell-year rows', 'value': metrics['actual_row_count']},
    {'check': 'duplicate analytical keys', 'value': metrics['duplicate_analytical_key_count']},
    {'check': 'panel SHA-256', 'value': metrics['panel_sha256']},
])
assert metrics['expected_row_count'] == metrics['actual_row_count']
assert metrics['duplicate_analytical_key_count'] == 0
display(panel_summary)

,check,value
0,canonical grid cells,89112
1,expected cell-year rows,891120
2,actual cell-year rows,891120
3,duplicate analytical keys,0
4,panel SHA-256,96C24EE6A4F5F6F5E06963CE97434AE22742AA6190A17D...
